# Task 58: Feedforward neural network

Reference: https://builtin.com/data-science/feedforward-neural-network-intro



In [1]:
# Task 58: Feedforward neural network
# Reference: https://builtin.com/data-science/feedforward-neural-network-intro

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import numpy as np
import tensorflow as tf

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(13)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x, y = make_moons(n_samples=400, noise=0.20, random_state=13)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, stratify=y, random_state=13)
scaler = StandardScaler().fit(x_train)
x_train = scaler.transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(2)
])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
print('Architecture: 2 inputs -> Dense(16) -> Dense(8) -> 2 logits')
print('Trainable parameters:', model.count_params())
print('Training/test samples:', len(x_train), len(x_test))

# Full-batch updates keep the small training example easy to inspect.
for epoch in range(1, 201):
    with tf.GradientTape() as tape:
        logits = model(x_train, training=True)
        loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
            y_train, logits, from_logits=True))
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    if epoch in [1, 50, 100, 200]:
        print(f'Epoch {epoch}: training loss={float(loss.numpy()):.4f}')

test_logits = model(x_test, training=False)
test_predictions = np.argmax(test_logits.numpy(), axis=1)
test_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
    y_test, test_logits, from_logits=True))
print('Test loss:', round(float(test_loss.numpy()), 4))
print('Test accuracy:', round(float(np.mean(test_predictions == y_test)), 4))
print('First ten predictions:', test_predictions[:10])


Architecture: 2 inputs -> Dense(16) -> Dense(8) -> 2 logits
Trainable parameters: 202
Training/test samples: 300 100
Epoch 1: training loss=0.6494
Epoch 50: training loss=0.2230
Epoch 100: training loss=0.0713
Epoch 200: training loss=0.0380
Test loss: 0.0312
Test accuracy: 0.98
First ten predictions: [0 0 1 1 0 0 0 1 1 1]
